# 03 Modeling

Train and evaluate pre-point serve outcome models with match-level cross-validation.

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, log_loss
from sklearn.model_selection import LeaveOneGroupOut

from src.train_model import PRE_POINT_FEATURES, evaluate_with_logo, build_preprocessor
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression


In [ ]:
df = pd.read_csv(Path('../data/processed/serves_cleaned.csv'))
df = df.dropna(subset=['match_id','point_win']).copy()
X = df[[c for c in PRE_POINT_FEATURES if c in df.columns]]
y = df['point_win']
groups = df['match_id']


In [ ]:
# Baseline: majority class
logo = LeaveOneGroupOut()
y_true, y_prob = [], []
for tr, te in logo.split(X,y,groups):
    clf = DummyClassifier(strategy='prior')
    clf.fit(X.iloc[tr], y.iloc[tr])
    p = clf.predict_proba(X.iloc[te])[:,1]
    y_true.extend(y.iloc[te].tolist())
    y_prob.extend(p.tolist())

y_pred = [1 if p>=0.5 else 0 for p in y_prob]
baseline_metrics = {
    'model':'baseline_prior',
    'accuracy':accuracy_score(y_true,y_pred),
    'roc_auc':roc_auc_score(y_true,y_prob),
    'precision':precision_score(y_true,y_pred,zero_division=0),
    'recall':recall_score(y_true,y_pred,zero_division=0),
    'log_loss':log_loss(y_true,y_prob),
}
baseline_metrics

In [ ]:
results = evaluate_with_logo(df)
results_df = pd.DataFrame([r.__dict__ for r in results])
results_df = pd.concat([pd.DataFrame([baseline_metrics]), results_df], ignore_index=True)
results_df

## Notes
- This notebook intentionally evaluates only pre-point predictors.
- Post-point fields are excluded to avoid leakage in serve recommendation use-cases.
